## Backend-Plugin mit OpenAI

Das Backend-Plugin wird mit dem offiziellen `backend-plugin`-Template erzeugt.

Es stellt den Endpunkt `GET /api/openai-tab/query` bereit.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn new --select backend-plugin --option pluginId=openai-tab

Dann darin das OpenAI Plugin installieren

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn workspace @internal/backstage-plugin-openai-tab-backend add openai

Der Code erstellt einen Express-Router für das Backend-Plugin, damit das Plugin einen eigenen HTTP-Endpunkt bereitstellen kann. 

Ein Aufruf von `/api/openai-tab/query` liefert die Antwort.


In [ ]:
%%bash
cd ~/mybackstage/
cat > plugins/openai-tab-backend/src/router.ts <<'EOF'
import { HttpAuthService } from '@backstage/backend-plugin-api';
import express, { Router } from 'express';
import { OpenAI } from 'openai';
import { LoggerService } from '@backstage/backend-plugin-api';

type RouterOptions = {
  logger: LoggerService;
};

export function createRouter({ httpAuth, logger }: RouterOptions) {
  const router = Router();
  router.use(express.json()); 

  const client = new OpenAI({ apiKey: process.env.OPENAI_API_KEY });

  router.post('/query', async (req, res) => {
    const { vectorStoreName, query } = req.body;

    logger.info(`Empfange Anfrage an /query: ${vectorStoreName} | ${query}`);

    try {
      const vsList = await client.vectorStores.list({ limit: 100 });
      const store = vsList.data.find(
        vs => vs.name.toLowerCase() === vectorStoreName.toLowerCase(),
      );

      if (!store) {
        return res.status(404).json({ error: 'Vector Store nicht gefunden' });
      }

      const response = await client.responses.create({
        model: 'gpt-4o-mini',
        input: query,
        tools: [
          {
            type: 'file_search',
            vector_store_ids: [store.id],
          },
        ],
      });

      const output = response.output?.find(item =>
        item.type === 'message',
      )?.content?.find(c => c.type === 'output_text');

      return res.json({ text: output?.text || 'Keine Antwort erhalten.' });
    } catch (err: any) {
      logger.error('Fehler beim OpenAI-Zugriff', err);
      return res.status(500).json({ error: 'Fehler beim OpenAI-Zugriff' });
    }
  });

  return router;
}
EOF


Der Code registriert das Ping-Plugin im Backstage-Backend, bindet den zuvor definierten Router ein und erlaubt den Zugriff auf `/ping` ohne Anmeldung. 

Die Datei `index.ts` exportiert das Plugin als Standardeinstiegspunkt, damit Backstage es laden kann.


In [ ]:
%%bash
cd ~/mybackstage/
cat > plugins/openai-tab-backend/src/plugin.ts <<'EOF'
import {
  coreServices,
  createBackendPlugin,
} from '@backstage/backend-plugin-api';

import { createRouter } from './router';

export const openaiTabPlugin = createBackendPlugin({
  pluginId: 'openai-tab',

  register(env) {
    env.registerInit({
      deps: {
        logger: coreServices.logger,
        httpAuth: coreServices.httpAuth,
        httpRouter: coreServices.httpRouter,
      },

      async init({ logger, httpRouter, httpAuth }) {
        httpRouter.addAuthPolicy({
          path: '/query',
          allow: 'unauthenticated',
        });

        httpRouter.use(
          createRouter({
            httpAuth,
            logger,
          }),
        );
      },
    });
  },
});
EOF

cat > plugins/openai-tab-backend/src/index.ts <<'EOF'
export { openaiTabPlugin as default } from './plugin';
EOF


Das CLI-Template trägt das Backend-Plugin normalerweise bereits in
`packages/backend/src/index.ts` ein. Die folgende Ausgabe dient nur zur Kontrolle.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
grep "plugin-openai-tab" packages/backend/src/index.ts

**Backstage starten:** Backstage wird mit dem neuen Plugin gestartet.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage OpenAI"
export BACKSTAGE_PORT="3001"

echo "http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn start --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.test.yaml


**Testen**

In einen separaten Terminal:

    curl -X POST \
      http://localhost:7007/api/openai-tab/query \
      -H "Content-Type: application/json" \
      -d '{
        "vectorStoreName": "cna",
        "query": "Welche Informationen befinden sich in den Dokumenten?"
      }'